# Decision Tree
This notebook implements Decision Tree concepts to classify the wine dataset by cultivar class.

## 1. Setup

We import the core scientific Python stack alongside `sklearn`'s model selection, tree, metrics, and inspection modules. `StratifiedKFold` is used throughout to ensure every cross-validation fold preserves the original class proportions — essential on a small dataset like Wine (178 samples across three classes). `permutation_importance` is imported now but used later as a model-agnostic alternative to the tree's built-in Gini importance.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from collections import Counter
from pprint import pprint

from sklearn.datasets import load_wine
from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    GridSearchCV,
    cross_validate,
    cross_val_predict,
)
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    auc,
    ConfusionMatrixDisplay,
)
from sklearn.inspection import permutation_importance

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.figsize"] = (10, 6)

---

## 2. Load the Wine Dataset

`load_wine()` returns a 178 × 13 numeric feature matrix and a 1D integer target vector. We wrap both in pandas objects so that column names are always visible in outputs. `class_mapping` maps the integer labels (0, 1, 2) to their human-readable cultivar names for display purposes. The class distribution print confirms whether the dataset is balanced — a prerequisite for using macro-averaged metrics later.

In [ ]:
data = load_wine()

X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name="target")

target_names = data.target_names
class_mapping = {i: name for i, name in enumerate(target_names)}

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)
print("\nClass distribution:")
print(y.value_counts().rename(index=class_mapping))

display(X.head())
display(X.describe().T)

---

## 3. Exploratory Data Analysis

Two plots give us a quick read on the data's structure before any modelling:

- **Class distribution** — checks for imbalance. Heavily skewed classes would require `class_weight='balanced'` or stratified sampling.
- **Correlation heatmap** — identifies pairs of highly correlated features. Decision trees handle correlated features gracefully (they simply ignore redundant ones), but strong correlations can make feature importance scores misleading.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.countplot(x=y.map(class_mapping), ax=axes[0], palette="Set2")
axes[0].set_title("Class Distribution")
axes[0].set_xlabel("Cultivar")
axes[0].set_ylabel("Count")

corr = X.corr()
sns.heatmap(corr, cmap="coolwarm", center=0, ax=axes[1], cbar=True)
axes[1].set_title("Feature Correlation Heatmap")

plt.tight_layout()
plt.show()

### Pairwise Feature View

We select the five features that most visually separate the three cultivars (flavanoids, proline, color intensity, OD ratio, and alcohol) and plot all pairwise combinations. The KDE diagonal shows each feature's distribution per class. Clear separation in any panel indicates that feature will likely appear near the root of the trained tree.

In [ ]:
top_features = [
    "flavanoids",
    "proline",
    "color_intensity",
    "od280/od315_of_diluted_wines",
    "alcohol",
]

pair_df = X[top_features].copy()
pair_df["target"] = y.map(class_mapping)

sns.pairplot(pair_df, hue="target", corner=True, diag_kind="kde")
plt.suptitle("Pairwise View of Selected Features", y=1.02)
plt.show()

---

## 4. Train / Test Split

We hold out **20%** of the data as a final test set that will not be seen during training or hyperparameter selection. `stratify=y` ensures both splits contain the same class proportions as the full dataset — without this, a random split on 178 samples could easily under-represent one cultivar.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    stratify=y,
    random_state=42,
)

print("Training set:", X_train.shape, y_train.shape)
print("Test set:    ", X_test.shape,  y_test.shape)

---

## 5. Hyperparameter Grid Validation

Before running an expensive grid search, we define a validation function that checks every value in the proposed parameter grid against `sklearn`'s accepted types and ranges. Catching a `ccp_alpha=-0.01` or a `max_depth=0` before the search starts avoids obscure runtime errors mid-way through thousands of fits.

The function also performs a **cross-parameter sanity check**: if `min_samples_split < 2 × min_samples_leaf` for any combination, a warning is printed. This situation over-constrains valid splits and can cause the tree to degenerate.

In [ ]:
def validate_decision_tree_param_grid(param_grid):
    """
    Validate a param grid for DecisionTreeClassifier.
    Raises ValueError with detailed messages if issues are found.
    """
    allowed_criteria          = {"gini", "entropy", "log_loss"}
    allowed_splitters         = {"best", "random"}
    allowed_max_features_str  = {"sqrt", "log2", None}
    allowed_ccp_alpha_min     = 0.0
    allowed_min_impurity_min  = 0.0

    if not isinstance(param_grid, dict):
        raise ValueError("param_grid must be a dictionary.")

    if len(param_grid) == 0:
        raise ValueError("param_grid cannot be empty.")

    for key, values in param_grid.items():
        if not isinstance(values, (list, tuple, np.ndarray)):
            raise ValueError(f"Values for '{key}' must be a list, tuple, or ndarray.")
        if len(values) == 0:
            raise ValueError(f"Values for '{key}' cannot be empty.")

        if key == "criterion":
            invalid = [v for v in values if v not in allowed_criteria]
            if invalid:
                raise ValueError(f"Invalid criterion values: {invalid}. Allowed: {allowed_criteria}")

        elif key == "splitter":
            invalid = [v for v in values if v not in allowed_splitters]
            if invalid:
                raise ValueError(f"Invalid splitter values: {invalid}. Allowed: {allowed_splitters}")

        elif key == "max_depth":
            for v in values:
                if v is not None and (not isinstance(v, int) or v <= 0):
                    raise ValueError(f"Invalid max_depth={v}. Must be None or a positive integer.")

        elif key == "min_samples_split":
            for v in values:
                valid_int   = isinstance(v, int)   and v >= 2
                valid_float = isinstance(v, float) and 0.0 < v <= 1.0
                if not (valid_int or valid_float):
                    raise ValueError(f"Invalid min_samples_split={v}. Must be int >= 2 or float in (0,1].")

        elif key == "min_samples_leaf":
            for v in values:
                valid_int   = isinstance(v, int)   and v >= 1
                valid_float = isinstance(v, float) and 0.0 < v <= 0.5
                if not (valid_int or valid_float):
                    raise ValueError(f"Invalid min_samples_leaf={v}. Must be int >= 1 or float in (0,0.5].")

        elif key == "max_features":
            for v in values:
                valid_int   = isinstance(v, int)   and v >= 1
                valid_float = isinstance(v, float) and 0.0 < v <= 1.0
                valid_str   = v in allowed_max_features_str
                if not (valid_int or valid_float or valid_str):
                    raise ValueError(
                        f"Invalid max_features={v}. Must be int >= 1, float in (0,1], "
                        f"or one of {allowed_max_features_str}."
                    )

        elif key == "class_weight":
            for v in values:
                if v is not None and v != "balanced":
                    raise ValueError(f"Invalid class_weight={v}. Allowed: None or 'balanced'.")

        elif key == "ccp_alpha":
            for v in values:
                if not isinstance(v, (int, float)) or v < allowed_ccp_alpha_min:
                    raise ValueError(f"Invalid ccp_alpha={v}. Must be a nonnegative number.")

        elif key == "min_impurity_decrease":
            for v in values:
                if not isinstance(v, (int, float)) or v < allowed_min_impurity_min:
                    raise ValueError(f"Invalid min_impurity_decrease={v}. Must be a nonnegative number.")

        elif key == "random_state":
            for v in values:
                if v is not None and not isinstance(v, int):
                    raise ValueError(f"Invalid random_state={v}. Must be None or int.")

    # Cross-parameter sanity check
    if "min_samples_split" in param_grid and "min_samples_leaf" in param_grid:
        for s in param_grid["min_samples_split"]:
            for l in param_grid["min_samples_leaf"]:
                if isinstance(s, int) and isinstance(l, int) and s < 2 * l:
                    print(
                        f"Warning: min_samples_split={s} < 2 * min_samples_leaf={2*l}. "
                        "This may overly constrain valid splits."
                    )

    print("Hyperparameter grid validation passed.")

---

## 6. Define the Hyperparameter Grid

We search over eight parameters simultaneously:

| Parameter | What it controls |
|:----------|:-----------------|
| `criterion` | Impurity measure used to evaluate candidate splits (Gini, entropy, or log-loss) |
| `splitter` | Whether to find the globally best split at each node (`best`) or a random one (`random`) |
| `max_depth` | Maximum tree depth; the primary lever for controlling overfitting |
| `min_samples_split` | Minimum samples required to attempt splitting an internal node |
| `min_samples_leaf` | Minimum samples required at each leaf; acts as implicit pruning |
| `max_features` | Number of features considered at each split; adds randomness similar to Random Forests |
| `class_weight` | `balanced` reweights classes inversely proportional to their frequency |
| `ccp_alpha` | Cost-complexity pruning strength applied after the tree is grown |

This grid contains roughly **9,200 unique combinations**. `GridSearchCV` evaluates each one with 5-fold cross-validation, so approximately **46,000 individual fits** are performed in total.

In [ ]:
param_grid = {
    "criterion":         ["gini", "entropy", "log_loss"],
    "splitter":          ["best", "random"],
    "max_depth":         [2, 3, 4, 5, 6, 8, 10, None],
    "min_samples_split": [2, 5, 10, 20],
    "min_samples_leaf":  [1, 2, 4, 8],
    "max_features":      [None, "sqrt", "log2"],
    "class_weight":      [None, "balanced"],
    "ccp_alpha":         [0.0, 0.001, 0.005, 0.01],
}

validate_decision_tree_param_grid(param_grid)
pprint(param_grid)

---

## 7. Cross-Validation Strategy and Scoring

`StratifiedKFold(n_splits=5)` splits the training data into five folds while preserving the class distribution in each fold. With only ~142 training samples, unstratified folds could accidentally place nearly all examples of one class in a single fold.

We score each fold on four metrics. All three classification metrics use **macro averaging**, which weights each class equally regardless of support — appropriate here because the three cultivar classes are roughly balanced.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Note: precision/recall/f1 need averaging for multiclass.
# We use macro averaging — treats all three cultivar classes equally.
scoring = {
    "accuracy":  "accuracy",
    "precision": "precision_macro",
    "recall":    "recall_macro",
    "f1":        "f1_macro",
}

---

## 8. Baseline Cross-Validation

Before tuning anything, we evaluate a **default, untuned** `DecisionTreeClassifier` using 5-fold stratified CV. This establishes a reference point against which the tuned model will be compared. `return_train_score=True` also returns training scores so we can detect overfitting — a large gap between train and test score is a signal that the tree is memorising the training data.

In [ ]:
baseline_model = DecisionTreeClassifier(random_state=42)

baseline_cv = cross_validate(
    baseline_model,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    return_train_score=True,
    n_jobs=-1,
)

baseline_results = pd.DataFrame(baseline_cv)
display(baseline_results)

print("Baseline CV performance:")
for metric in scoring.keys():
    mean_score = baseline_results[f"test_{metric}"].mean()
    std_score  = baseline_results[f"test_{metric}"].std()
    print(f"{metric:>10}: mean={mean_score:.4f}, std={std_score:.4f}")

### Baseline CV Score Distribution

We melt the per-fold scores into long format so seaborn can produce grouped boxplots. The **boxplot** shows the distribution across folds while the **strip plot** overlays the raw fold scores. Wide boxes or large spreads indicate high variance — the model's performance depends heavily on which samples land in the fold.

In [ ]:
baseline_long = baseline_results[[f"test_{m}" for m in scoring.keys()]].copy()
baseline_long.columns = [c.replace("test_", "") for c in baseline_long.columns]
baseline_long = baseline_long.melt(var_name="Metric", value_name="Score")

plt.figure(figsize=(12, 6))
sns.boxplot(data=baseline_long,  x="Metric", y="Score", palette="pastel")
sns.stripplot(data=baseline_long, x="Metric", y="Score", color="black", alpha=0.6)
plt.title("Baseline Decision Tree: Cross-Validation Metric Distribution")
plt.show()

---

## 9. Grid Search with Cross-Validation

`GridSearchCV` performs an exhaustive search over the parameter grid, evaluating each combination with 5-fold stratified CV. We optimise for **F1-macro** rather than accuracy because F1 jointly captures precision and recall, and macro averaging ensures no single class dominates.

`refit=True` (the default) re-trains the best configuration on the **full training set** after the search completes, making `grid_search.best_estimator_` immediately ready for prediction.

In [ ]:
grid_search = GridSearchCV(
    estimator=DecisionTreeClassifier(random_state=42),
    param_grid=param_grid,
    scoring="f1_macro",
    cv=cv,
    n_jobs=-1,
    verbose=1,
    refit=True,
    return_train_score=True,
)

grid_search.fit(X_train, y_train)

print("Best parameters:")
pprint(grid_search.best_params_)
print(f"\nBest CV F1 (macro): {grid_search.best_score_:.4f}")

best_model = grid_search.best_estimator_

### Top 20 Hyperparameter Configurations

We sort all evaluated configurations by their mean CV F1 score and visualise the top 20. The bar length represents the mean F1 across the five folds for that configuration. Configurations with similar bar lengths are roughly equivalent — examining the corresponding parameter settings (in `sorted_results`) can reveal which parameters matter most.

In [ ]:
results_df   = pd.DataFrame(grid_search.cv_results_)
sorted_results = results_df.sort_values("mean_test_score", ascending=False).reset_index(drop=True)

top_n = 20
top_results = sorted_results.head(top_n).copy()

plt.figure(figsize=(14, 8))
sns.barplot(
    data=top_results,
    x="mean_test_score",
    y=top_results.index.astype(str),
    orient="h",
    palette="viridis",
)
plt.title(f"Top {top_n} Hyperparameter Settings by Mean CV F1 (macro)")
plt.xlabel("Mean CV F1 (macro)")
plt.ylabel("Ranked Configuration Index")
plt.show()

---

## 10. Tuned Model Cross-Validation

We re-run the same 5-fold CV on the best model using the same `cv` object and `scoring` dictionary as the baseline. This guarantees an apples-to-apples comparison: both models are evaluated on the same folds with the same metrics. The baseline mean is printed alongside each tuned score so the improvement is immediately visible.

In [ ]:
tuned_cv = cross_validate(
    best_model,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    return_train_score=True,
    n_jobs=-1,
)

tuned_results = pd.DataFrame(tuned_cv)

print("Tuned model CV performance:")
for metric in scoring.keys():
    base_mean = baseline_results[f"test_{metric}"].mean()
    tune_mean = tuned_results[f"test_{metric}"].mean()
    tune_std  = tuned_results[f"test_{metric}"].std()
    print(f"{metric:>10}: mean={tune_mean:.4f}, std={tune_std:.4f}  (baseline={base_mean:.4f})")

### Baseline vs. Tuned: Side-by-Side CV Comparison

The grouped boxplot overlays the fold-score distributions for both models across all four metrics. Ideally the tuned model's boxes are shifted upward (higher scores) and narrowed (lower variance), indicating both better performance and more consistent generalisation.

In [ ]:
compare_rows = []
for metric in scoring.keys():
    for fold_score in baseline_results[f"test_{metric}"]:
        compare_rows.append({"Metric": metric, "Score": fold_score, "Model": "Baseline"})
    for fold_score in tuned_results[f"test_{metric}"]:
        compare_rows.append({"Metric": metric, "Score": fold_score, "Model": "Tuned"})

compare_df = pd.DataFrame(compare_rows)

plt.figure(figsize=(13, 6))
sns.boxplot(data=compare_df, x="Metric", y="Score", hue="Model", palette="Set2")
plt.title("Baseline vs. Tuned Decision Tree: CV Score Distribution")
plt.legend(loc="lower right")
plt.show()

---

## 11. Test Set Evaluation

The test set has been untouched until now. We fit the best model on the full training set and evaluate it on the held-out 20% for a single, unbiased performance estimate. `classification_report` gives per-class precision, recall, and F1 alongside support (number of true instances per class), making it easy to see if the model struggles with any specific cultivar.

In [ ]:
best_model.fit(X_train, y_train)
y_pred = best_model.predict(X_test)

print("=" * 50)
print("Test Set Evaluation – Best Decision Tree")
print("=" * 50)
print(classification_report(y_test, y_pred, target_names=target_names))

### Confusion Matrix

The confusion matrix shows the count of each (actual, predicted) class pair. Rows represent the **true** class; columns represent the **predicted** class. Cells on the diagonal are correct predictions; off-diagonal cells are misclassifications. This makes it easy to identify which cultivar pairs are most commonly confused.

In [ ]:
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=target_names)

fig, ax = plt.subplots(figsize=(7, 6))
disp.plot(ax=ax, colorbar=True, cmap="Blues")
ax.set_title("Confusion Matrix – Tuned Decision Tree (Test Set)")
plt.show()

---

## 12. Decision Tree Structure

`plot_tree` renders the full tree diagram. Each internal node shows:

- The **split condition** (feature name and threshold)
- The **impurity value** (Gini or entropy) at that node
- The **sample count** passing through
- The **class distribution** (value array) of those samples

Leaf nodes show the **predicted class** for any sample that reaches them. Nodes are **colour-coded** by their majority class, with deeper colour indicating higher purity.

In [ ]:
fig, ax = plt.subplots(figsize=(24, 10))

plot_tree(
    best_model,
    feature_names=X.columns.tolist(),
    class_names=target_names,
    filled=True,
    rounded=True,
    impurity=True,
    ax=ax,
)

plt.title("Tuned Decision Tree Structure")
plt.tight_layout()
plt.show()

---

## 13. Feature Importances

### Gini Importance (Mean Decrease in Impurity)

The tree's built-in `feature_importances_` attribute reports how much each feature reduced the weighted impurity across all nodes it was used to split, normalised so all importances sum to 1. Features with a value of 0 were never selected for a split.

> **Caveat:** Gini importance is computed from the training data and can be biased toward features with many unique values (high cardinality), because they offer more candidate thresholds to exploit.

In [ ]:
importances = best_model.feature_importances_
importance_df = pd.DataFrame({
    "feature":    X.columns,
    "importance": importances,
}).sort_values("importance", ascending=False).reset_index(drop=True)

plt.figure(figsize=(10, 7))
sns.barplot(data=importance_df, x="importance", y="feature", palette="viridis")
plt.title("Feature Importances – Tuned Decision Tree")
plt.xlabel("Gini Importance")
plt.ylabel("Feature")
plt.show()

display(importance_df)

### Permutation Importance

Permutation importance is a **model-agnostic** alternative that directly measures a feature's predictive contribution on the **test set**. For each feature, its values are randomly shuffled 30 times — breaking the relationship between that feature and the target — and the resulting drop in accuracy is recorded.

A large mean drop means the feature carries genuine signal. A drop near zero (or negative) means the model performs just as well without it. Error bars show the standard deviation across the 30 shuffles.

Comparing Gini and permutation rankings highlights whether any features are over-valued by the impurity criterion but don't actually help generalise.

In [ ]:
perm = permutation_importance(
    best_model, X_test, y_test,
    n_repeats=30,
    random_state=42,
    n_jobs=-1,
)

perm_df = pd.DataFrame({
    "feature": X.columns,
    "mean":    perm.importances_mean,
    "std":     perm.importances_std,
}).sort_values("mean", ascending=False).reset_index(drop=True)

plt.figure(figsize=(10, 7))
plt.barh(perm_df["feature"], perm_df["mean"], xerr=perm_df["std"], color="steelblue")
plt.axvline(0, linewidth=1)
plt.xlabel("Mean accuracy decrease")
plt.title("Permutation Importances – Tuned Decision Tree (Test Set)")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

display(perm_df)

---

## 14. Bias–Variance Trade-off: Depth Sweep

We sweep `max_depth` from 1 to 15 (plus unconstrained `None`), holding all other hyperparameters fixed at the tuned values from the grid search. This isolates depth's contribution to the bias–variance trade-off:

- **Shallow trees** (low depth) underfit — both train and test accuracy are low (high bias).
- **Deep trees** (high depth or None) overfit — train accuracy is high but test accuracy drops (high variance).
- The **optimal depth** balances the two, which should correspond closely to the value chosen by the grid search.

In [ ]:
depths = list(range(1, 16)) + [None]
train_scores, test_scores = [], []

for d in depths:
    m = DecisionTreeClassifier(
        criterion=best_model.criterion,
        min_samples_split=best_model.min_samples_split,
        min_samples_leaf=best_model.min_samples_leaf,
        max_features=best_model.max_features,
        class_weight=best_model.class_weight,
        ccp_alpha=best_model.ccp_alpha,
        max_depth=d,
        random_state=42,
    )
    m.fit(X_train, y_train)
    train_scores.append(accuracy_score(y_train, m.predict(X_train)))
    test_scores.append(accuracy_score(y_test,  m.predict(X_test)))

depth_labels = [str(d) if d is not None else "None" for d in depths]

plt.figure(figsize=(11, 5))
plt.plot(depth_labels, train_scores, marker="o", label="Train Accuracy")
plt.plot(depth_labels, test_scores,  marker="s", label="Test Accuracy")
plt.axvline(
    depth_labels.index(str(best_model.max_depth) if best_model.max_depth is not None else "None"),
    linestyle="--", color="gray", label=f"Best depth ({best_model.max_depth})"
)
plt.xlabel("max_depth")
plt.ylabel("Accuracy")
plt.title("Decision Tree Accuracy vs. max_depth")
plt.legend()
plt.show()

### Cost-Complexity Pruning Path

Cost-complexity pruning is a post-hoc pruning strategy that removes subtrees whose accuracy improvement does not justify their added complexity. The trade-off is controlled by `ccp_alpha`:

$$\text{Score}_{\alpha}(T) = \text{Impurity}(T) + \alpha \cdot |T|$$

where $|T|$ is the number of leaves. `cost_complexity_pruning_path` returns the sequence of effective alphas at which each subtree is pruned. As `ccp_alpha` increases, the tree shrinks — test accuracy initially improves (pruning noise) then degrades (over-pruning).

In [ ]:
path = DecisionTreeClassifier(random_state=42).cost_complexity_pruning_path(X_train, y_train)
ccp_alphas = path.ccp_alphas[:-1]  # exclude the trivial last value

ccp_train, ccp_test = [], []
for alpha in ccp_alphas:
    m = DecisionTreeClassifier(ccp_alpha=alpha, random_state=42)
    m.fit(X_train, y_train)
    ccp_train.append(accuracy_score(y_train, m.predict(X_train)))
    ccp_test.append(accuracy_score(y_test,  m.predict(X_test)))

plt.figure(figsize=(11, 5))
plt.plot(ccp_alphas, ccp_train, marker="o", label="Train Accuracy")
plt.plot(ccp_alphas, ccp_test,  marker="s", label="Test Accuracy")
plt.xlabel("ccp_alpha (pruning strength)")
plt.ylabel("Accuracy")
plt.title("Cost-Complexity Pruning: Accuracy vs. Alpha")
plt.legend()
plt.show()

---

## 15. Final Summary

The summary table collects the key metrics for both the baseline and tuned models in one place. The baseline is **re-fitted from scratch** here to ensure no state from earlier cells contaminates the comparison. The best hyperparameter configuration and the top three features by Gini importance are printed below the table.

In [ ]:
summary_rows = [
    {
        "Model":    "Baseline Decision Tree",
        "CV F1 (macro) mean": baseline_results["test_f1"].mean(),
        "CV F1 (macro) std":  baseline_results["test_f1"].std(),
        "Test Accuracy": accuracy_score(y_test, DecisionTreeClassifier(random_state=42).fit(X_train, y_train).predict(X_test)),
        "Test F1 (macro)": f1_score(y_test, DecisionTreeClassifier(random_state=42).fit(X_train, y_train).predict(X_test), average="macro"),
    },
    {
        "Model":    "Tuned Decision Tree",
        "CV F1 (macro) mean": tuned_results["test_f1"].mean(),
        "CV F1 (macro) std":  tuned_results["test_f1"].std(),
        "Test Accuracy": accuracy_score(y_test, y_pred),
        "Test F1 (macro)": f1_score(y_test, y_pred, average="macro"),
    },
]

summary_df = pd.DataFrame(summary_rows).set_index("Model")
display(summary_df.round(4))

print(f"\nBest hyperparameters:")
pprint(grid_search.best_params_)

top3 = importance_df.head(3)["feature"].tolist()
print(f"\nTop 3 features by Gini importance: {top3}")